In [77]:
# --- Production-Ready Error Handling and Logging Example ---
import logging
import pandas as pd

logging.basicConfig(level=logging.INFO, format='%(asctime)s %(levelname)s %(message)s')

def safe_read_csv(path):
    try:
        df = pd.read_csv(path)
        logging.info(f"Loaded {path} successfully. Shape: {df.shape}")
        return df
    except Exception as e:
        logging.error(f"Failed to load {path}: {e}")
        return None

# Example usage:
# customers_df = safe_read_csv('dataset/cleandata/customers.csv')
# if customers_df is not None:
#     # proceed with processing

In [78]:
# Importing Necessary Libraries
import pandas as pd


In [79]:
# Loading the dataset
yanki_df = pd.read_csv('dataset/rawdata/yanki_ecommerce.csv')
# yanki_df.head()

In [80]:
# Getting information about the dataset
# yanki_df.info()

In [81]:
# Drop missing values
yanki_df.dropna(subset=["Order_ID", "Customer_ID"], inplace=True)
# yanki_df.info()

In [82]:
# Convert Order_Date from string to datetime object
yanki_df["Order_Date"] = pd.to_datetime(yanki_df["Order_Date"], errors="coerce")
# yanki_df.info()

In [83]:
# Data Cleaning and Normalization

# customers table
customers_df = (
    yanki_df[["Customer_ID", "Customer_Name", "Email", "Phone_Number"]]
    .copy()
    .drop_duplicates()
    .reset_index(drop=True)
)

# products table
products_df = (
    yanki_df[["Product_ID", "Product_Name", "Brand", "Category", "Price"]]
    .copy()
    .drop_duplicates()
    .reset_index(drop=True)
)

# Shipping Address table
shipping_address_df = (
    yanki_df[
        ["Customer_ID", "Shipping_Address", "City", "State", "Country", "Postal_Code"]
    ]
    .copy()
    .drop_duplicates()
    .reset_index(drop=True)
)

# Creating shipping address ID
shipping_address_df.index.name = "Shipping_ID"
shipping_address_df.reset_index(inplace=True)

# Orders table
orders_df = (
    yanki_df[
        [
            "Order_ID",
            "Customer_ID",
            "Product_ID",
            "Quantity",
            "Total_Price",
            "Order_Date",
        ]
    ]
    .copy()
    .drop_duplicates()
    .reset_index(drop=True)
)

# Payment method table
payment_method_df = (
    yanki_df[["Order_ID", "Payment_Method", "Transaction_Status"]]
    .copy()
    .drop_duplicates()
    .reset_index(drop=True)
)

# customers_df.head()
# products_df.head()
# shipping_address_df.head()
# orders_df.head()
# payment_method_df.head()

In [84]:
# Save the cleaned and normalized data to new CSV files

customers_df.to_csv("dataset/cleandata//customers.csv", index=False)
products_df.to_csv("dataset/cleandata//products.csv", index=False)
shipping_address_df.to_csv("dataset/cleandata//shipping_address.csv", index=False)
orders_df.to_csv("dataset/cleandata//orders.csv", index=False)
payment_method_df.to_csv("dataset/cleandata//payment_method.csv", index=False)

In [85]:
from psycopg2.extensions import ISOLATION_LEVEL_AUTOCOMMIT

from dotenv import load_dotenv
import os
import psycopg2

# Load environment variables from .env file
load_dotenv()

# Get database connection variables from environment
db_host = os.getenv("DB_HOST")
db_port = os.getenv("DB_PORT")
db_name = os.getenv("DB_NAME")
db_user = os.getenv("DB_USER")
db_password = os.getenv("DB_PASSWORD")


def create_database_if_not_exists(dbname, user, password, host, port):
    # Connect to the default database
    conn = psycopg2.connect(
        dbname=db_name, user=db_user, password=db_password, host=db_host, port=db_port
    )
    conn.set_isolation_level(ISOLATION_LEVEL_AUTOCOMMIT)
    cur = conn.cursor()
    # Check if database exists
    cur.execute(f"SELECT 1 FROM pg_database WHERE datname = %s;", (dbname,))
    exists = cur.fetchone()
    if not exists:
        cur.execute(f'CREATE DATABASE "{dbname}";')
        print(f"Database '{dbname}' created.")
    else:
        print(f"Database '{dbname}' already exists.")
    cur.close()
    conn.close()


# Usage
create_database_if_not_exists(
    dbname=db_name, user=db_user, password=db_password, host=db_host, port=db_port
)

Database 'windowcase' already exists.


In [86]:
from dotenv import load_dotenv
import os
import psycopg2

# Load environment variables from .env file
load_dotenv()

# Get database connection variables from environment
db_host = os.getenv("DB_HOST")
db_name = os.getenv("DB_NAME")
db_user = os.getenv("DB_USER")
db_password = os.getenv("DB_PASSWORD")


def get_db_connection():
    connection = psycopg2.connect(
        host=db_host,
        database=db_name,
        user=db_user,
        password=db_password,
    )
    return connection

In [87]:
# Create our SQL tables
def create_tables():
    conn = get_db_connection()
    cursor = conn.cursor()

    statements = [
        "CREATE SCHEMA IF NOT EXISTS yanki;",
        "DROP TABLE IF EXISTS yanki.payment_method CASCADE;",
        "DROP TABLE IF EXISTS yanki.orders CASCADE;",
        "DROP TABLE IF EXISTS yanki.shipping_address CASCADE;",
        "DROP TABLE IF EXISTS yanki.products CASCADE;",
        "DROP TABLE IF EXISTS yanki.customers CASCADE;",
        """
        CREATE TABLE IF NOT EXISTS yanki.customers (
            Customer_ID UUID PRIMARY KEY,
            Customer_Name TEXT,
            Email TEXT,
            Phone_Number TEXT
        );
        """,
        """
        CREATE TABLE IF NOT EXISTS yanki.products (
            Product_ID UUID PRIMARY KEY,
            Product_Name TEXT,
            Brand TEXT,
            Category TEXT,
            Price FLOAT
        );
        """,
        """
        CREATE TABLE IF NOT EXISTS yanki.shipping_address (
            shipping_ID INTEGER PRIMARY KEY,
            Customer_ID UUID,
            Shipping_Address TEXT,
            City TEXT,
            State TEXT,
            Country TEXT,
            Postal_Code INTEGER,
            FOREIGN KEY (Customer_ID) REFERENCES yanki.customers(Customer_ID)
        );
        """,
        """
        CREATE TABLE IF NOT EXISTS yanki.orders (
            Order_ID UUID PRIMARY KEY,
            Customer_ID UUID,
            Product_ID UUID,
            Quantity INTEGER,
            Total_Price FLOAT,
            Order_Date DATE,
            FOREIGN KEY (Customer_ID) REFERENCES yanki.customers(Customer_ID),
            FOREIGN KEY (Product_ID) REFERENCES yanki.products(Product_ID)
        );
        """,
        """
        CREATE TABLE IF NOT EXISTS yanki.payment_method (
            Order_ID UUID,
            Payment_Method TEXT,
            Transaction_Status TEXT,
            FOREIGN KEY (Order_ID) REFERENCES yanki.orders(Order_ID)
        );
        """,
    ]

    for sql in statements:
        cursor.execute(sql)

    conn.commit()
    cursor.close()
    conn.close()


create_tables()
print("Schema and tables created successfully.")

Schema and tables created successfully.


In [88]:
import csv

def load_data_from_csv(csv_path):
    conn = get_db_connection()
    cursor = conn.cursor()

    # Ensure target schema/table exist before inserting rows
    cursor.execute("CREATE SCHEMA IF NOT EXISTS yanki;")
    cursor.execute("""
        CREATE TABLE IF NOT EXISTS yanki.customers (
            Customer_ID UUID PRIMARY KEY,
            Customer_Name TEXT,
            Email TEXT,
            Phone_Number TEXT
        );
        """)

    with open(csv_path, "r") as file:
        reader = csv.reader(file)
        next(reader)
        for row in reader:
            cursor.execute(
                """
                INSERT INTO yanki.customers (Customer_ID, Customer_Name, Email, Phone_Number)
                VALUES (%s, %s, %s, %s)
                ON CONFLICT (Customer_ID) DO NOTHING;
                """,
                row,
            )

    conn.commit()
    cursor.close()
    conn.close()


csv_file_path = "dataset/cleandata/customers.csv"

load_data_from_csv(csv_file_path)
print("Customer data loaded into yanki.customers.")

Customer data loaded into yanki.customers.


In [89]:
import csv


def load_data_from_csv(csv_path):
    conn = get_db_connection()
    cursor = conn.cursor()
    with open(csv_path, "r") as file:
        reader = csv.reader(file)
        next(reader)
        for row in reader:
            cursor.execute(
                """
                            INSERT INTO yanki.products (Product_ID, Product_Name, Brand, Category, Price)
                            VALUES (%s, %s, %s, %s, %s);""",
                row,
            )
    conn.commit()
    cursor.close()
    conn.close()


csv_file_path = "dataset/cleandata/products.csv"

load_data_from_csv(csv_file_path)

In [90]:
import csv


def load_data_from_csv(csv_path):
    conn = get_db_connection()
    cursor = conn.cursor()
    with open(csv_path, "r") as file:
        reader = csv.reader(file)
        next(reader)
        for row in reader:
            cursor.execute(
                """
                            INSERT INTO yanki.shipping_address (shipping_ID, Customer_ID, Shipping_Address, City, State, Country, Postal_Code)
                            VALUES (%s, %s, %s, %s, %s, %s, %s);""",
                row,
            )
    conn.commit()
    cursor.close()
    conn.close()


csv_file_path = "dataset/cleandata/shipping_address.csv"

load_data_from_csv(csv_file_path)

In [91]:
import csv

def load_data_from_csv(csv_path):
    conn = get_db_connection()
    cursor = conn.cursor()
    with open(csv_path, "r") as file:
        reader = csv.reader(file)
        next(reader)
        for row in reader:
            cursor.execute(
                """
                            INSERT INTO yanki.orders (Order_ID, Customer_ID, Product_ID, Quantity, Total_Price, Order_Date)
                            VALUES (%s, %s, %s, %s, %s, %s);""",
                row,
            )
    conn.commit()
    cursor.close()
    conn.close()


csv_file_path = "dataset/cleandata/orders.csv"

load_data_from_csv(csv_file_path)

In [92]:
import csv

def load_data_from_csv(csv_path):
    conn = get_db_connection()
    cursor = conn.cursor()
    with open(csv_path, "r") as file:
        reader = csv.reader(file)
        next(reader)
        for row in reader:
            cursor.execute(
                """
                            INSERT INTO yanki.payment_method (Order_ID, Payment_Method, Transaction_Status)
                            VALUES (%s, %s, %s);""",
                row,
            )
    conn.commit()
    cursor.close()
    conn.close()


csv_file_path = "dataset/cleandata/payment_method.csv"

load_data_from_csv(csv_file_path)